# Clase 7 — Normales y reconstrucción de superficies

**Procesamiento Geométrico y Análisis de Formas** · Ivan Sipiran · DCC, Universidad de Chile

---

Este tutorial cubre los dos temas de la clase:

1. **Estimación de normales** en nubes de puntos: ajuste de plano por mínimos cuadrados
   totales, elección del tamaño de vecindad, orientación consistente y eliminación de
   outliers.
2. **Reconstrucción de superficies**: representaciones implícitas, marching squares /
   marching cubes, y los tres métodos clásicos — Hoppe (plano tangente), RBF (Carr) y
   Poisson (Kazhdan).

**Duración estimada: 120 minutos.**

| Bloque | Contenido | Ejercicios | Min |
|---|---|---|---|
| 0 | Preparación: superficie de referencia, muestreo, visualización | — | 8 |
| 1.1 | Plano por mínimos cuadrados totales (multiplicadores de Lagrange) | 1 | 10 |
| 1.2 | Normales por PCA local, vectorizado | 2 | 8 |
| 1.3 | Tamaño de vecindad: ruido vs curvatura, y $k^*(\sigma)$ | 3 | 10 |
| 1.4 | Orientación consistente por árbol de expansión mínima | 4 | 9 |
| 1.5 | Outliers y la razón de autovalores | 5 | 8 |
| 2.0 | Superficies implícitas: por qué funcionan | — | 4 |
| 2.1 | Marching squares: tabla de casos, interpolación, ambigüedad | 6 | 10 |
| 2.2 | Marching cubes / marching tetrahedra | 7 | 11 |
| 2.3 | Reconstrucción de Hoppe (plano tangente) | 8 | 8 |
| 2.4 | Superficies implícitas por RBF (Carr et al.) | 9 | 8 |
| 2.5 | Reconstrucción de Poisson (Kazhdan et al.) | 10 | 13 |
| 2.6 | Comparación: ruido, huecos, densidad. Point2Mesh | — | 5 |
| 3 | Contraste con Open3D | — | 5 |
| 4 | Cierre, preguntas de discusión, bibliografía | — | 3 |

**Reglas del juego.** Todo se implementa desde cero con NumPy / SciPy / Matplotlib: no se
descarga ningún dato ni se instala nada, salvo en el Bloque 3, donde se compara contra
Open3D. Cada ejercicio es una celda con `TODO` seguida de una celda de tests con `assert`.
Las soluciones están al final del notebook, en celdas plegadas: si te trabas, ejecuta la
solución correspondiente (redefine la función) y sigue adelante.

---
# Bloque 0 — Preparación

Necesitamos datos con **groundtruth**. En vez de descargar un modelo, definimos una
superficie *analítica* como el conjunto de nivel cero de una función implícita
$F:\mathbb{R}^3 \to \mathbb{R}$ construida con *metaballs*:

$$F(\mathbf{x}) \;=\; t \;-\; \sum_{i=1}^{m} w_i \, \exp\!\left(-\frac{\lVert \mathbf{x}-\mathbf{c}_i\rVert^2}{s_i^2}\right)$$

**Convención que usaremos en todo el notebook: $F < 0$ adentro, $F > 0$ afuera.** Con eso
$\nabla F$ apunta hacia afuera y la normal exacta en cualquier punto de la superficie es
$\mathbf{n} = \nabla F / \lVert \nabla F \rVert$.

Esto nos da tres cosas gratis:

* muestras exactamente sobre la superficie (proyección de Newton),
* la **normal verdadera** en cada muestra, para medir error angular,
* la **distancia exacta a la superficie** de cualquier punto reconstruido, vía
  $d(\mathbf{x}) \approx |F(\mathbf{x})| / \lVert\nabla F(\mathbf{x})\rVert$ (primer orden),
  para medir el error de reconstrucción.

Los centros están dispuestos en un anillo, así que la superficie tiene **genus 1**: hay un
agujero. Eso lo vamos a usar más adelante para verificar que la malla reconstruida tiene la
topología correcta.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.dpi'] = 110

# ---------------------------------------------------------------- superficie
_C = np.array([[ 0.90,  0.00,  0.00], [ 0.45,  0.78,  0.00],
               [-0.45,  0.78,  0.05], [-0.90,  0.00,  0.00],
               [-0.45, -0.78, -0.05], [ 0.45, -0.78,  0.00],
               [ 0.60,  0.35,  0.45],                 # rompen la simetria
               [-0.30, -0.55, -0.40]])
_W = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.55, 0.45])
_S = np.array([0.55, 0.55, 0.55, 0.55, 0.55, 0.55, 0.38, 0.34])
_T = 0.60

def F(P):
    "Funcion implicita de referencia.  F < 0 dentro,  F > 0 fuera."
    P = np.atleast_2d(np.asarray(P, float))
    d2 = ((P[:, None, :] - _C[None, :, :]) ** 2).sum(-1)
    return _T - (_W * np.exp(-d2 / _S ** 2)).sum(1)

def gradF(P):
    "Gradiente analitico de F (apunta hacia afuera)."
    P = np.atleast_2d(np.asarray(P, float))
    D = P[:, None, :] - _C[None, :, :]
    g = _W * np.exp(-(D ** 2).sum(-1) / _S ** 2) * (2.0 / _S ** 2)
    return (g[:, :, None] * D).sum(1)

def dist_to_surface(P):
    "Distancia (a primer orden) de cada punto a la superficie F = 0."
    return np.abs(F(P)) / np.linalg.norm(gradF(P), axis=1)

In [ ]:
# ------------------------------------------------- muestreo de la superficie
def project_to_surface(P, iters=25):
    "Newton a lo largo del gradiente:  x <- x - F(x) grad F / |grad F|^2."
    P = P.copy()
    for _ in range(iters):
        g = gradF(P)
        P -= (F(P) / np.maximum((g ** 2).sum(1), 1e-12))[:, None] * g
    return P

def sample_surface(n, seed=0, relax=25, box=1.8):
    "Muestreo casi uniforme de F = 0: proyeccion + relajacion tangencial."
    rng = np.random.default_rng(seed)
    P = project_to_surface(rng.uniform(-box, box, size=(4 * n, 3)))
    P = P[(np.abs(F(P)) < 1e-8) & (np.abs(P).max(1) < box)]
    P = P[rng.permutation(len(P))[:n]]
    for _ in range(relax):                       # repulsion entre vecinos
        d, j = cKDTree(P).query(P, k=9)
        h = np.median(d[:, 1])
        V = P[:, None, :] - P[j[:, 1:], :]
        r = np.linalg.norm(V, axis=2) + 1e-12
        w = np.maximum(0.0, 1.0 - r / (2.2 * h))
        step = (w[:, :, None] * V / r[:, :, None]).sum(1) * 0.35 * h
        g = gradF(P); g /= np.linalg.norm(g, axis=1, keepdims=True)
        P = project_to_surface(P + step - (step * g).sum(1)[:, None] * g, 6)
    N = gradF(P); N /= np.linalg.norm(N, axis=1, keepdims=True)
    return P, N                                  # normales VERDADERAS

def add_noise(P, N, sigma, seed=0):
    "Ruido gaussiano a lo largo de la normal (modelo de escaneo)."
    rng = np.random.default_rng(seed)
    return P + sigma * rng.standard_normal((len(P), 1)) * N

def add_outliers(P, frac, box=1.5, seed=0):
    rng = np.random.default_rng(seed)
    O = rng.uniform(-box, box, size=(int(frac * len(P)), 3))
    lab = np.concatenate([np.zeros(len(P), bool), np.ones(len(O), bool)])
    p = rng.permutation(len(P) + len(O))
    return np.vstack([P, O])[p], lab[p]

def angle_err(N, Ngt):
    "Error angular en grados, SIN considerar el signo (normales no orientadas)."
    return np.degrees(np.arccos(np.abs((N * Ngt).sum(1)).clip(0, 1)))

In [ ]:
# -------------------------------------------------------------- visualizacion
_L = np.array([-0.35, -0.75, 0.55]); _L /= np.linalg.norm(_L)

def _rot(az, el):
    a, e = np.radians(az), np.radians(el)
    Rz = np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1]])
    Rx = np.array([[1, 0, 0], [0, np.cos(e), -np.sin(e)], [0, np.sin(e), np.cos(e)]])
    return Rx @ Rz

def shade(ax, V, N=None, az=35, el=20, s=4.0, cmap='bone', c=None,
          cull=True, vmin=None, vmax=None, title=None):
    """Render barato: proyeccion ortografica + sombreado lambertiano +
    back-face culling. Si se pasa `c`, colorea por ese escalar."""
    R = _rot(az, el); Q = V @ R.T
    if N is not None and cull:
        Nr = N @ R.T; vis = Nr[:, 1] < 0
    else:
        vis = np.ones(len(V), bool)
    if c is None:
        lam = np.clip(-((N @ R.T) @ _L), 0, 1) if N is not None else np.ones(len(V))
        col, vmin, vmax = 0.20 + 0.80 * lam, 0.0, 1.0
    else:
        col = np.asarray(c, float)
    o = np.argsort(-Q[vis, 1])
    im = ax.scatter(Q[vis, 0][o], Q[vis, 2][o], c=col[vis][o], s=s, cmap=cmap,
                    vmin=vmin, vmax=vmax, linewidths=0, marker='o')
    ax.set_aspect('equal'); ax.axis('off')
    if title: ax.set_title(title, fontsize=10)
    return im

def quiver3(ax, P, N, az=35, el=20, scale=0.10, every=1, color='crimson', **kw):
    R = _rot(az, el); Q = P @ R.T; M = N @ R.T
    m = slice(None, None, every)
    ax.quiver(Q[m, 0], Q[m, 2], M[m, 0] * scale, M[m, 2] * scale,
              color=color, angles='xy', scale_units='xy', scale=1,
              width=0.003, **kw)

In [ ]:
P_gt, N_gt = sample_surface(8000, seed=1)
DIAG = np.linalg.norm(P_gt.max(0) - P_gt.min(0))
print('%d puntos   diagonal del bbox = %.3f   |F| maximo = %.1e'
      % (len(P_gt), DIAG, np.abs(F(P_gt)).max()))
d, _ = cKDTree(P_gt).query(P_gt, k=2)
print('distancia al vecino mas cercano: media %.4f  (desv/media = %.2f)'
      % (d[:, 1].mean(), d[:, 1].std() / d[:, 1].mean()))

fig, axs = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
shade(axs[0], P_gt, N_gt, s=5, title='nube limpia (8000 pts)')
shade(axs[1], P_gt, N_gt, s=5, az=90, el=88, title='vista superior: genus 1')
shade(axs[2], add_noise(P_gt, N_gt, 0.02, 7), N_gt, s=5,
      title=r'con ruido $\sigma$ = 0.02')
plt.show()

---
# Bloque 1 — Estimación de normales

## 1.1 El plano por mínimos cuadrados totales

Suponemos un buen muestreo de la superficie. En un entorno pequeño de un punto
$\mathbf{p}$, la superficie es aproximadamente un plano; queremos el plano que mejor
aproxima a los vecinos $\{\mathbf{q}_1,\dots,\mathbf{q}_k\}$, y su normal.

Un plano es $\{\mathbf{x} : \mathbf{n}^\top(\mathbf{x}-\mathbf{c}) = 0\}$ con
$\lVert\mathbf{n}\rVert = 1$. La distancia de $\mathbf{q}_i$ al plano es
$|\mathbf{n}^\top(\mathbf{q}_i - \mathbf{c})|$, así que minimizamos

$$E(\mathbf{c}, \mathbf{n}) = \sum_{i=1}^{k} \left(\mathbf{n}^\top(\mathbf{q}_i-\mathbf{c})\right)^2
\qquad \text{sujeto a} \quad \lVert \mathbf{n}\rVert^2 = 1 .$$

Anulando $\partial E/\partial \mathbf{c}$ se obtiene que $\mathbf{c}$ es el **centroide**
$\bar{\mathbf{q}}$. Sustituyendo, $E(\mathbf{n}) = \mathbf{n}^\top C\, \mathbf{n}$ con la
matriz de covarianza

$$C \;=\; \frac{1}{k}\sum_{i=1}^{k} (\mathbf{q}_i-\bar{\mathbf{q}})(\mathbf{q}_i-\bar{\mathbf{q}})^\top \in \mathbb{R}^{3\times 3}.$$

Con el multiplicador de Lagrange $\lambda$ para la restricción,
$\mathcal{L} = \mathbf{n}^\top C \mathbf{n} - \lambda(\mathbf{n}^\top\mathbf{n}-1)$, y

$$\nabla_\mathbf{n}\mathcal{L} = 2C\mathbf{n} - 2\lambda\mathbf{n} = 0 \;\Longrightarrow\; C\mathbf{n} = \lambda \mathbf{n}.$$

Es decir: **$\mathbf{n}$ es un autovector de $C$**, y como el valor del objetivo en ese
punto es $\mathbf{n}^\top C\mathbf{n} = \lambda$, el mínimo se alcanza en el autovector del
**menor** autovalor. Los otros dos autovectores generan el plano tangente.

Nota: esto **no** es lo mismo que ajustar $z = ax+by+c$ por mínimos cuadrados ordinarios.
Ese modelo minimiza el residuo vertical, privilegia el eje $z$ y se degenera cuando el
plano es casi vertical. Lo vamos a comprobar.

### Ejercicio 1 — Ajuste de plano por mínimos cuadrados totales

In [ ]:
def fit_plane(Q):
    c = Q.mean(0)
    Y = Q - c
    C = (Y.T @ Y) / len(Q)
    w, V = np.linalg.eigh(C)          # w ascendente, V columnas
    return c, V[:, 0], w

In [ ]:
# --- Tests del ejercicio 1 ---
rng = np.random.default_rng(0)
# plano VERTICAL (su normal no tiene componente z) con ruido isotropico
n_true = np.array([1.0, 0.15, 0.0]); n_true /= np.linalg.norm(n_true)
e1, e2 = np.linalg.svd(n_true[None, :])[2][1:]
Q = (rng.uniform(-1, 1, (200, 1)) * e1 + rng.uniform(-1, 1, (200, 1)) * e2
     + np.array([0.3, -0.2, 0.7]))
Qn = Q + 0.01 * rng.standard_normal((200, 3))

c, n, w = fit_plane(Qn)
err = np.degrees(np.arccos(abs(n @ n_true)))
print('error angular TLS: %.3f grados' % err)
print('autovalores: %.2e %.2e %.2e' % tuple(w))
assert err < 1.0, 'el ajuste TLS deberia recuperar la normal con < 1 grado'
assert w[0] < w[1] < w[2], 'los autovalores deben venir en orden ascendente'
assert abs(np.linalg.norm(n) - 1) < 1e-9, 'la normal debe ser unitaria'
assert np.allclose(c, Qn.mean(0)), 'c debe ser el centroide'

# contraste: minimos cuadrados ordinarios z = a x + b y + c
A = np.c_[Qn[:, 0], Qn[:, 1], np.ones(len(Qn))]
sol, *_ = np.linalg.lstsq(A, Qn[:, 2], rcond=None)
n_ols = np.array([-sol[0], -sol[1], 1.0]); n_ols /= np.linalg.norm(n_ols)
print('error angular OLS (z = ax+by+c): %.1f grados  <-- se degenera'
      % np.degrees(np.arccos(abs(n_ols @ n_true))))
print('OK')

El ajuste vertical falla porque el plano es casi paralelo al eje $z$: el modelo
$z=ax+by+c$ no puede representarlo. El TLS no privilegia ningún eje.

## 1.2 Normales por PCA local

La receta para toda la nube: para cada punto, tomar sus $k$ vecinos más cercanos y aplicar
1.1. Hacerlo con un `for` sobre 8000 puntos es lento; conviene armar el tensor de vecindades
de una vez y usar `np.linalg.eigh` sobre un lote de matrices $3\times 3$ (soporta entrada
con *batch*: `(n,3,3) -> (n,3), (n,3,3)`).

Las covarianzas por lote salen con `np.einsum('nki,nkj->nij', Y, Y)`, donde `Y` es el tensor
de vecindades centradas de forma `(n,k,3)`.

### Ejercicio 2 — Normales por PCA local, vectorizado

In [ ]:
def normals_pca(P, k=20):
    _, idx = cKDTree(P).query(P, k=k)
    Y = P[idx]
    Y = Y - Y.mean(1, keepdims=True)
    C = np.einsum('nki,nkj->nij', Y, Y) / k
    w, V = np.linalg.eigh(C)
    return V[:, :, 0], w

In [ ]:
# --- Tests del ejercicio 2 ---
import time
t0 = time.time(); N_est, W = normals_pca(P_gt, k=20); dt = time.time() - t0
e = angle_err(N_est, N_gt)
print('%.2f s para %d puntos' % (dt, len(P_gt)))
print('error angular: mediana %.3f  media %.3f  p95 %.3f grados'
      % (np.median(e), e.mean(), np.percentile(e, 95)))
assert N_est.shape == P_gt.shape and W.shape == P_gt.shape
assert np.allclose(np.linalg.norm(N_est, axis=1), 1), 'normales unitarias'
assert np.median(e) < 1.5, 'la mediana del error deberia bajar de 1.5 grados'
assert dt < 5.0, 'demasiado lento: probablemente quedo un bucle sobre puntos'
print('OK')

fig, axs = plt.subplots(1, 2, figsize=(9.5, 4.2), constrained_layout=True)
shade(axs[0], P_gt, N_est, s=5, title='normales estimadas (k=20)')
im = shade(axs[1], P_gt, N_est, s=5, c=e, cmap='inferno', vmin=0, vmax=3,
           title='error angular [grados]')
plt.colorbar(im, ax=axs[1], shrink=0.7); plt.show()

El error no está repartido de forma uniforme: se concentra donde la superficie se dobla
más. Ese es exactamente el sesgo por curvatura que estudiamos ahora.

## 1.3 El tamaño de la vecindad

Dos efectos compiten (Mitra & Nguyen, 2003). Con vecindad de radio $r$, densidad $\rho$,
ruido $\sigma$ y curvatura $\kappa$:

$$\underbrace{\Delta\theta_{\text{ruido}} \;\sim\; \frac{\sigma}{r\sqrt{\rho r^2}} \;\sim\; \frac{\sigma}{k}}_{\text{varianza: baja al crecer } k}
\qquad\qquad
\underbrace{\Delta\theta_{\text{curv}} \;\sim\; \kappa\, r \;\sim\; \kappa \sqrt{k/\rho}}_{\text{sesgo: sube al crecer } k}$$

usando $k \approx \rho \pi r^2$, o sea $r \propto \sqrt{k}$. En escala log-log esperamos
pendiente $-1$ en la rama de ruido y $+1/2$ en la de curvatura. Minimizando la suma,
$k^* \propto \sigma^{2/3}$.

Ojo con los extremos: cuando el error angular llega a las decenas de grados la aproximación
lineal deja de valer y la rama de ruido **satura** (una normal no puede errar más de 90°),
así que las pendientes hay que medirlas en el régimen moderado.

La conclusión práctica es la de la slide: **el $k$ óptimo depende de la curvatura local, que
no conocemos**. Vamos a medirlo.

### Ejercicio 3 — Barrido de vecindad: las dos ramas y $k^*(\sigma)$

In [ ]:
def sweep_k(P, N_true, sigmas, ks, seed=11):
    E = np.zeros((len(sigmas), len(ks)))
    for i, s in enumerate(sigmas):
        Pn = P if s == 0 else add_noise(P, N_true, s, seed)
        for j, k in enumerate(ks):
            E[i, j] = np.median(angle_err(normals_pca(Pn, k)[0], N_true))
    return E

In [ ]:
# --- Tests del ejercicio 3 ---
sigmas = [0.0, 0.005, 0.01, 0.02, 0.04]
ks = [6, 8, 10, 14, 20, 28, 40, 60, 90, 130]
E = sweep_k(P_gt, N_gt, sigmas, ks)

print('sigma \\ k ', ' '.join('%6d' % k for k in ks))
for i, s in enumerate(sigmas):
    print('%8.3f  ' % s, ' '.join('%6.2f' % v for v in E[i]),
          '  -> k* = %d' % ks[int(np.argmin(E[i]))])

kstar = np.array([ks[int(np.argmin(row))] for row in E])
assert np.all(np.diff(kstar) >= 0), 'k* deberia crecer (no decrecer) con el ruido'
assert kstar[-1] > kstar[1], 'con mas ruido conviene una vecindad mas grande'

lk = np.log(ks)
# rama de ruido: k chico y ruido moderado (sigma = 0.01, la fila 2). Con
# sigma = 0.04 y k = 6 el error ya satura cerca de los 30 grados y la
# pendiente se aplana.
p_noise = np.polyfit(lk[:4], np.log(E[2, :4]), 1)[0]
# rama de curvatura: sin ruido, k grande
p_curv = np.polyfit(lk[5:], np.log(E[0, 5:]), 1)[0]
print('\npendiente log-log rama de ruido    : %+.2f   (teoria: -1)' % p_noise)
print('pendiente log-log rama de curvatura: %+.2f   (teoria: +0.5)' % p_curv)
assert -1.25 < p_noise < -0.80, 'la rama de ruido deberia caer como 1/k'
assert 0.25 < p_curv < 0.75, 'la rama de curvatura deberia subir como sqrt(k)'
print('OK')

fig, ax = plt.subplots(figsize=(6, 4.2), constrained_layout=True)
for i, s in enumerate(sigmas):
    ax.loglog(ks, E[i], 'o-', ms=4, label=r'$\sigma$ = %.3f' % s)
kk = np.array(ks, float)
ax.loglog(kk, E[2, 0] * (kk / kk[0]) ** -1.0, 'k--', lw=1, label=r'$k^{-1}$')
ax.loglog(kk, E[0, -1] * (kk / kk[-1]) ** 0.5, 'k:', lw=1, label=r'$k^{1/2}$')
ax.set_xlabel('k (vecinos)'); ax.set_ylabel('error angular mediano [grados]')
ax.legend(fontsize=8); ax.grid(alpha=0.3, which='both'); plt.show()

La curva en forma de U es el contenido de la slide 14: con poco ruido conviene una vecindad
chica (el sesgo por curvatura domina), con mucho ruido conviene una grande. Y como $\kappa$
varía sobre la superficie, un $k$ global es siempre un compromiso. De ahí las estrategias
adaptativas: elegir $r$ por punto según una estimación local de curvatura y densidad.

## 1.4 Orientación consistente

El PCA entrega la normal salvo el signo: $\pm\mathbf{n}$ minimizan lo mismo. Para
reconstruir una superficie necesitamos un campo de normales **orientado** de forma
consistente (todas hacia afuera). El algoritmo clásico es de Hoppe et al. (1992):

1. Construir el grafo de $k$ vecinos más cercanos.
2. Ponderar la arista $(i,j)$ con $w_{ij} = 1 - |\mathbf{n}_i^\top\mathbf{n}_j|$: cuesta poco
   propagar entre normales casi paralelas, mucho entre normales casi perpendiculares.
3. Calcular el **árbol de expansión mínima** de ese grafo. El MST prefiere propagar por las
   zonas planas y evita saltar entre dos hojas de superficie que están cerca en el espacio
   pero lejos sobre la superficie.
4. Recorrer el árbol desde una raíz cuya orientación conocemos (ej.: el punto más alto,
   cuya normal apunta hacia $+z$) y voltear $\mathbf{n}_i$ si $\mathbf{n}_i^\top\mathbf{n}_{\text{padre}} < 0$.

El paso 3 es el que importa: propagar por el vecino más cercano sin MST falla justo donde
dos partes de la superficie se acercan.

### Ejercicio 4 — Orientación consistente por árbol de expansión mínima

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import minimum_spanning_tree, breadth_first_order

def orient_normals(P, N, k=12):
    n = len(P)
    _, idx = cKDTree(P).query(P, k=k + 1)
    rows = np.repeat(np.arange(n), k)
    cols = idx[:, 1:].ravel()
    w = 1.0 - np.abs((N[rows] * N[cols]).sum(1)) + 1e-6
    T = minimum_spanning_tree(coo_matrix((w, (rows, cols)), shape=(n, n)))
    root = int(np.argmax(P[:, 2]))
    order, pred = breadth_first_order(T + T.T, root, directed=False)
    N = N.copy()
    if N[root, 2] < 0:
        N[root] *= -1
    for i in order[1:]:
        p = pred[i]
        if p >= 0 and N[i] @ N[p] < 0:
            N[i] *= -1
    return N

In [ ]:
# --- Tests del ejercicio 4 ---
P_n = add_noise(P_gt, N_gt, 0.005, seed=3)
N_raw, _ = normals_pca(P_n, k=30)
rng = np.random.default_rng(0)
N_raw = N_raw * rng.choice([-1.0, 1.0], len(N_raw))[:, None]   # signos al azar
print('antes de orientar: %.1f%% coincide con la normal verdadera'
      % (100 * ((N_raw * N_gt).sum(1) > 0).mean()))

N_or = orient_normals(P_n, N_raw, k=12)
frac = ((N_or * N_gt).sum(1) > 0).mean()
print('despues de orientar: %.1f%%   error angular mediano %.2f grados'
      % (100 * frac, np.median(angle_err(N_or, N_gt))))
assert np.allclose(np.abs((N_or * N_raw).sum(1)), 1, atol=1e-9), \
    'solo se permite cambiar el signo, no la direccion'
assert frac > 0.999, 'la orientacion no quedo globalmente consistente'
print('OK')

fig, axs = plt.subplots(1, 2, figsize=(9.5, 4.2), constrained_layout=True)
shade(axs[0], P_n, N_raw, cull=False, s=5,
      c=((N_raw * N_gt).sum(1) > 0).astype(float), cmap='coolwarm',
      title='signos al azar')
shade(axs[1], P_n, N_or, cull=False, s=5,
      c=((N_or * N_gt).sum(1) > 0).astype(float), cmap='coolwarm',
      title='tras propagar por el MST')
plt.show()

## 1.5 Outliers, y lo que dicen los autovalores

La slide agrupa "estimación de normales y eliminación de outliers" porque se resuelven con
el mismo objeto: la **vecindad local**. Dos herramientas:

**Filtro estadístico.** Para cada punto, la distancia media a sus $k$ vecinos. En una
superficie bien muestreada esa distancia es homogénea; un outlier flotando en el aire tiene
vecinos lejos. Se descarta lo que supere $\mu + \alpha\,\sigma$ de esa distribución.

**Razón de autovalores.** De $\lambda_0 \le \lambda_1 \le \lambda_2$ de la covarianza local
salen descriptores útiles:

$$\text{variación de superficie } \; \sigma_\lambda = \frac{\lambda_0}{\lambda_0+\lambda_1+\lambda_2}, \qquad
\text{linealidad } \frac{\lambda_2-\lambda_1}{\lambda_2}, \qquad
\text{planaridad } \frac{\lambda_1-\lambda_0}{\lambda_2}.$$

$\sigma_\lambda \approx 0$ en una zona plana, crece con la curvatura y se dispara ($\to 1/3$,
el valor isotrópico) donde la vecindad es una nube de ruido sin estructura. Es la
"curvatura de superficie" de Pauly et al. (2002), y también un detector de outliers.

### Ejercicio 5 — Filtro estadístico de outliers y variación de superficie

In [ ]:
def statistical_outlier_mask(P, k=16, alpha=1.0):
    d, _ = cKDTree(P).query(P, k=k + 1)
    md = d[:, 1:].mean(1)
    return md < md.mean() + alpha * md.std()

def surface_variation(W):
    return W[:, 0] / np.maximum(W.sum(1), 1e-300)

In [ ]:
# --- Tests del ejercicio 5 ---
P_out, is_out = add_outliers(P_gt, 0.06, seed=5)
keep = statistical_outlier_mask(P_out, k=16, alpha=1.0)
prec = is_out[~keep].mean() if (~keep).sum() else 0.0
rec = (~keep)[is_out].mean()
print('%d outliers de %d puntos' % (is_out.sum(), len(P_out)))
print('descartados %d  |  precision %.3f  recall %.3f' % ((~keep).sum(), prec, rec))
assert prec > 0.95, 'casi todo lo descartado deberia ser outlier'
assert rec > 0.75, 'deberia recuperar al menos el 75% de los outliers'

_, W_out = normals_pca(P_out, k=20)
sv = surface_variation(W_out)
print('variacion de superficie: inliers %.4f  outliers %.4f  (mediana)'
      % (np.median(sv[~is_out]), np.median(sv[is_out])))
assert np.median(sv[is_out]) > 5 * np.median(sv[~is_out]), \
    'los outliers deberian tener variacion de superficie mucho mayor'

_, W_clean = normals_pca(P_gt, k=20)
sv_clean = surface_variation(W_clean)
assert sv_clean.max() < 0.34 and sv_clean.min() >= 0
print('OK')

fig, axs = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)
shade(axs[0], P_out, None, s=4, cull=False, c=is_out.astype(float),
      cmap='coolwarm', title='nube con 6% de outliers')
shade(axs[1], P_out, None, s=4, cull=False, c=(~keep).astype(float),
      cmap='coolwarm', title='marcados por el filtro')
im = shade(axs[2], P_gt, N_gt, s=5, c=sv_clean, cmap='viridis', vmin=0,
           vmax=np.percentile(sv_clean, 99),
           title=r'variación de superficie $\sigma_\lambda$ (nube limpia)')
plt.colorbar(im, ax=axs[2], shrink=0.7); plt.show()

El mapa de $\sigma_\lambda$ reproduce la curvatura de la superficie: alto en los cuellos
entre bolas, bajo en las zonas planas. Es el mismo cálculo que hicimos para las normales,
leído distinto — de ahí que ambos problemas vayan juntos en la clase.

**Cierre del bloque 1.** Ya tenemos lo que necesita todo lo que sigue: una nube con normales
estimadas, limpias y orientadas. Ese es el objeto de entrada canónico de la reconstrucción
de superficies.

In [ ]:
# Entrada canonica para el Bloque 2: nube ruidosa, normales estimadas y orientadas.
P_in = add_noise(P_gt, N_gt, 0.005, seed=3)
keep = statistical_outlier_mask(P_in, k=16, alpha=2.0)
P_in = P_in[keep]
N_in = orient_normals(P_in, normals_pca(P_in, k=30)[0], k=12)
print('%d puntos | error angular mediano %.2f grados | %.1f%% bien orientadas'
      % (len(P_in), np.median(angle_err(N_in, N_gt[keep])),
         100 * ((N_in * N_gt[keep]).sum(1) > 0).mean()))

---
# Bloque 2 — Reconstrucción de superficies

## 2.0 Por qué una representación implícita

El objetivo es construir una malla poligonal a partir de la nube. El problema es que **los
datos no están estructurados**: los puntos no vienen ordenados ni conectados, y no sabemos
qué puntos son vecinos *sobre la superficie*. Peor: el problema está mal condicionado — por
un conjunto finito de puntos pasan infinitas superficies.

La estrategia dominante desde los 90 es indirecta:

$$\text{nube orientada} \;\longrightarrow\; \underbrace{f:\mathbb{R}^3\to\mathbb{R}}_{\text{campo escalar}} \;\longrightarrow\; \underbrace{\{ \mathbf{x} : f(\mathbf{x}) = 0\}}_{\text{isosuperficie}} \;\longrightarrow\; \text{malla}$$

Se ajusta un **campo escalar** en todo el espacio (o en una grilla que lo cubre) tal que la
superficie sea su conjunto de nivel cero, y después se extrae la isosuperficie. Ventajas:

* el ajuste es un problema de aproximación bien planteado (con regularización explícita),
* el resultado es automáticamente **cerrado y sin auto-intersecciones**,
* el ruido se promedia, y los huecos se rellenan solos por continuidad del campo,
* la topología (cuántas componentes, cuántos agujeros) sale del campo, no hay que decidirla.

Vamos a ver primero la segunda flecha (extracción de isosuperficie) y después las tres
maneras clásicas de construir $f$: Hoppe, RBF y Poisson.

## 2.1 Marching squares

Empezamos en 2D, donde todo se ve. Tenemos $f$ muestreada en una grilla regular y queremos
la curva $f=0$. El algoritmo:

1. **Tratar cada celda por separado.** Cada celda tiene 4 esquinas, cada una "dentro"
   ($f<0$) o "fuera" ($f\ge 0$): $2^4 = 16$ configuraciones.
2. **Enumerar los casos.** Codificamos la configuración en 4 bits. Con equivalencias de
   rotación y complemento, los 16 casos se reducen a 4 patrones: vacío, una esquina, un
   borde (dos esquinas adyacentes) y la silla de montar (dos esquinas opuestas).
3. **Interpolar el cruce** en cada arista que cambia de signo. Si la arista va de
   $\mathbf{a}$ (valor $v_a$) a $\mathbf{b}$ (valor $v_b$) con signos distintos, el cruce
   lineal está en

   $$\mathbf{x} = \mathbf{a} + t\,(\mathbf{b}-\mathbf{a}), \qquad t = \frac{v_a}{v_a - v_b}.$$

4. **Conectar** los cruces según la tabla.

Numeramos las esquinas de la celda en sentido antihorario: $0=(i,j)$, $1=(i{+}1,j)$,
$2=(i{+}1,j{+}1)$, $3=(i,j{+}1)$; y las aristas $e_0 = (0,1)$, $e_1 = (1,2)$, $e_2 = (2,3)$,
$e_3 = (3,0)$. El código es $\sum_c 2^c\,[\,f_c < 0\,]$.

In [ ]:
# Aristas de la celda y tabla de casos (dada).
MS_EDGE = {0: (0, 1), 1: (1, 2), 2: (2, 3), 3: (3, 0)}
MS_TABLE = {
    0: [],            15: [],
    1: [(3, 0)],      14: [(3, 0)],       # esquina 0
    2: [(0, 1)],      13: [(0, 1)],       # esquina 1
    4: [(1, 2)],      11: [(1, 2)],       # esquina 2
    8: [(3, 2)],       7: [(3, 2)],       # esquina 3
    3: [(3, 1)],      12: [(3, 1)],       # borde horizontal
    6: [(0, 2)],       9: [(0, 2)],       # borde vertical
    5: [(3, 0), (1, 2)],                  # silla de montar  <-- ambiguo
    10: [(0, 1), (2, 3)],                 # silla de montar  <-- ambiguo
}
print('16 casos, %d con contorno' % sum(1 for v in MS_TABLE.values() if v))

### Ejercicio 6 — Marching squares

In [ ]:
def marching_squares(G, xs, ys, iso=0.0):
    G = np.asarray(G, float) - iso
    nx, ny = G.shape
    XY = np.stack(np.meshgrid(xs, ys, indexing='ij'), -1)
    off = [(0, 0), (1, 0), (1, 1), (0, 1)]
    v = [G[a:nx - 1 + a, b:ny - 1 + b] for a, b in off]
    p = [XY[a:nx - 1 + a, b:ny - 1 + b] for a, b in off]

    code = sum((v[c] < 0) * (1 << c) for c in range(4))

    X = {}
    for e, (a, b) in MS_EDGE.items():
        t = v[a] / np.where(v[a] - v[b] == 0, 1e-300, v[a] - v[b])
        X[e] = p[a] + t[..., None] * (p[b] - p[a])

    segs = []
    for c, lst in MS_TABLE.items():
        if not lst:
            continue
        m = code == c
        if not m.any():
            continue
        for e0, e1 in lst:
            segs.append(np.stack([X[e0][m], X[e1][m]], 1))
    return np.concatenate(segs, 0) if segs else np.zeros((0, 2, 2))

In [ ]:
# --- Tests del ejercicio 6 ---
# Circulo de radio 1: el contorno debe medir 2 pi
xs = ys = np.linspace(-1.5, 1.5, 80)
Gc = np.hypot(*np.meshgrid(xs, ys, indexing='ij')) - 1.0
S = marching_squares(Gc, xs, ys)
L = np.linalg.norm(S[:, 1] - S[:, 0], axis=1).sum()
print('%d segmentos, largo %.4f  (2*pi = %.4f, error %.2f%%)'
      % (len(S), L, 2 * np.pi, 100 * abs(L - 2 * np.pi) / (2 * np.pi)))
assert abs(L - 2 * np.pi) / (2 * np.pi) < 0.005
r = np.linalg.norm(S.reshape(-1, 2), axis=1)
assert np.abs(r - 1).max() < 0.01, 'los vertices deben caer sobre el circulo'

# Corte z = 0 de nuestra superficie 3D
xs2 = np.linspace(-1.6, 1.6, 200); ys2 = np.linspace(-1.6, 1.6, 200)
XX, YY = np.meshgrid(xs2, ys2, indexing='ij')
Q0 = np.c_[XX.ravel(), YY.ravel(), np.zeros(XX.size)]
G0 = F(Q0).reshape(XX.shape)
S0 = marching_squares(G0, xs2, ys2)
print('corte z=0: %d segmentos' % len(S0))
assert len(S0) > 100
print('OK')

from matplotlib.collections import LineCollection
fig, axs = plt.subplots(1, 2, figsize=(10, 4.4), constrained_layout=True)
im = axs[0].contourf(XX, YY, G0, levels=21, cmap='RdBu_r',
                     vmin=-abs(G0).max(), vmax=abs(G0).max())
axs[0].add_collection(LineCollection(S0, colors='k', lw=1.5))
axs[0].set_aspect('equal'); axs[0].set_title('campo F en z = 0 y su nivel cero')
plt.colorbar(im, ax=axs[0], shrink=0.8)
axs[1].add_collection(LineCollection(S0, colors='k', lw=1.0))
axs[1].autoscale(); axs[1].set_aspect('equal')
axs[1].set_title('solo el contorno (2 componentes: exterior e interior)')
plt.show()

**Casos ambiguos.** Los códigos 5 y 10 tienen dos esquinas opuestas dentro y las otras dos
fuera. La tabla elige *una* de las dos formas de conectar los cruces (dos arcos separados),
pero la otra (dos arcos que se cruzan al revés) es igual de válida a partir de los datos: la
grilla simplemente no sabe si en el centro de la celda la superficie se junta o se separa.

En 2D el efecto es cosmético. En 3D **el mismo problema puede dejar agujeros en la malla**
si dos celdas vecinas resuelven la ambigüedad de forma incompatible en la cara que
comparten. Ese es el bug clásico del marching cubes original (1987), resuelto después con
las tablas extendidas de Nielson & Hamann (1991) y Chernyaev (1995).

In [ ]:
# La ambiguedad, en concreto: codigo 5 (dentro las esquinas 0 y 2).
corners = np.array([[0., 0.], [1., 0.], [1., 1.], [0., 1.]])
val = np.array([-1.0, 1.0, -1.0, 1.0])
X = {e: corners[a] + val[a] / (val[a] - val[b]) * (corners[b] - corners[a])
     for e, (a, b) in MS_EDGE.items()}

fig, axs = plt.subplots(1, 2, figsize=(8.5, 4.3), constrained_layout=True)
for ax, pares, tit, sub in [
        (axs[0], MS_TABLE[5], 'resolución A (la de la tabla)',
         'las dos esquinas "dentro" quedan SEPARADAS'),
        (axs[1], [(0, 1), (2, 3)], 'resolución B (igual de válida)',
         'las dos esquinas "dentro" quedan CONECTADAS')]:
    ax.add_collection(LineCollection([corners[[0, 1, 2, 3, 0]]], colors='0.7', lw=1))
    ax.add_collection(LineCollection([[X[a], X[b]] for a, b in pares],
                                     colors='crimson', lw=2.5))
    for c, v in zip(corners, val):
        ax.plot(*c, 'o', ms=11, color='steelblue' if v < 0 else 'white',
                mec='k', zorder=3)
    for p in X.values():
        ax.plot(*p, 's', ms=5, color='k', zorder=3)
    ax.set_xlim(-0.25, 1.25); ax.set_ylim(-0.25, 1.3)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(tit + '\n' + sub, fontsize=9)
plt.show()
print('Ambas conexiones son consistentes con los signos de las 4 esquinas:')
print('la grilla no tiene informacion para distinguirlas.')

## 2.2 Marching cubes y marching tetrahedra

En 3D la celda es un vóxel con 8 esquinas: $2^8 = 256$ configuraciones, que por simetrías
de rotación y complemento se reducen a **14 casos únicos** con triángulos (15 contando el
vóxel vacío), que es la tabla de la slide. El algoritmo
es idéntico: clasificar las esquinas, interpolar los cruces sobre las aristas del cubo y
conectar los triángulos según la tabla.

Escribir la tabla de 256 entradas a mano es tedioso y propenso a error, y además hay que
manejar las ambigüedades de cara. Existe una alternativa mucho más simple que da mallas
correctas **por construcción**: **marching tetrahedra**. Se descompone cada vóxel en 6
tetraedros que comparten la diagonal principal (descomposición de Freudenthal, consistente
entre vóxeles vecinos), y se extrae la isosuperficie tetraedro por tetraedro:

* un tetraedro tiene 4 vértices, o sea **16 casos**, no 256;
* con 1 vértice dentro (o 3), la superficie corta 3 aristas: **un triángulo**;
* con 2 vértices dentro, corta 4 aristas: **un cuadrilátero** (dos triángulos);
* **no hay casos ambiguos**: el complemento de un caso es otro caso ya enumerado, y las
  caras compartidas entre tetraedros vecinos se resuelven siempre igual.

El precio: unos 4× más triángulos que marching cubes y triángulos peor formados. En
producción se usa marching cubes; para entender el algoritmo, y para no depender de una
tabla de 256 filas, marching tetrahedra es el camino corto.

Numeramos las 6 aristas del tetraedro como
$e_0=(0,1),\, e_1=(0,2),\, e_2=(0,3),\, e_3=(1,2),\, e_4=(1,3),\, e_5=(2,3)$.

### Ejercicio 7 — Tabla de casos de marching tetrahedra

In [ ]:
TET_EDGES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]

def tet_case_table():
    def ei(x, y):
        return TET_EDGES.index((x, y) if (x, y) in TET_EDGES else (y, x))

    table = {}
    for code in range(16):
        ins = [i for i in range(4) if (code >> i) & 1]
        out = [i for i in range(4) if not (code >> i) & 1]
        if len(ins) in (0, 4):
            table[code] = []
        elif len(ins) in (1, 3):
            cross = [e for e, (a, b) in enumerate(TET_EDGES)
                     if (a in ins) != (b in ins)]
            table[code] = [tuple(cross)]
        else:
            a, b = ins
            c, d = out
            q = [ei(a, c), ei(a, d), ei(b, d), ei(b, c)]
            table[code] = [(q[0], q[1], q[2]), (q[0], q[2], q[3])]
    return table

In [ ]:
# --- Tests del ejercicio 7 ---
TET_TABLE = tet_case_table()
sizes = [len(TET_TABLE[c]) for c in range(16)]
print('triangulos por caso:', sizes)
assert sizes == [0, 1, 1, 2, 1, 2, 2, 1, 1, 2, 2, 1, 2, 1, 1, 0]

for code, trs in TET_TABLE.items():
    ins = [i for i in range(4) if (code >> i) & 1]
    cross = {e for e, (a, b) in enumerate(TET_EDGES) if (a in ins) != (b in ins)}
    used = {e for tr in trs for e in tr}
    assert used == cross, 'caso %d: usa aristas que no cambian de signo' % code
    if len(trs) == 2:
        # el quad debe ser un ciclo: aristas consecutivas comparten un vertice
        q = [trs[0][0], trs[0][1], trs[0][2], trs[1][2]]
        for i in range(4):
            e1, e2 = set(TET_EDGES[q[i]]), set(TET_EDGES[q[(i + 1) % 4]])
            assert len(e1 & e2) == 1, 'caso %d: el quad no esta en orden ciclico' % code
print('OK')

In [ ]:
# --- Dada la tabla, el resto es contabilidad. Esta funcion se entrega hecha.
CUBE_OFF = np.array([[0, 0, 0], [0, 0, 1], [0, 1, 0], [0, 1, 1],
                     [1, 0, 0], [1, 0, 1], [1, 1, 0], [1, 1, 1]])
# 6 tetraedros por voxel, todos conteniendo la diagonal 0-7 (Freudenthal)
TETS = np.array([[0, 7, 1, 3], [0, 7, 3, 2], [0, 7, 2, 6],
                 [0, 7, 6, 4], [0, 7, 4, 5], [0, 7, 5, 1]])

def marching_tets(G, axes, iso=0.0):
    """Isosuperficie G = iso.  Devuelve (V, T): vertices y triangulos."""
    G = np.asarray(G, float) - iso
    nx, ny, nz = G.shape
    grid = np.stack(np.meshgrid(*axes, indexing='ij'), -1)
    sl = [(slice(a, nx - 1 + a), slice(b, ny - 1 + b), slice(c, nz - 1 + c))
          for a, b, c in CUBE_OFF]
    val = np.stack([G[s] for s in sl], -1).reshape(-1, 8)
    pos = np.stack([grid[s] for s in sl], -2).reshape(-1, 8, 3)

    tris = []
    for tet in TETS:
        v, p = val[:, tet], pos[:, tet]
        code = sum((v[:, i] < 0) * (1 << i) for i in range(4))
        for c, trs in TET_TABLE.items():
            if not trs:
                continue
            m = code == c
            if not m.any():
                continue
            vm, pm = v[m], p[m]
            X = {}
            for e in {i for tr in trs for i in tr}:
                a, b = TET_EDGES[e]
                den = vm[:, a] - vm[:, b]
                t = vm[:, a] / np.where(np.abs(den) < 1e-300, 1e-300, den)
                X[e] = pm[:, a] + t[:, None] * (pm[:, b] - pm[:, a])
            cin = pm[:, [i for i in range(4) if (c >> i) & 1]].mean(1)
            for tr in trs:
                T = np.stack([X[tr[0]], X[tr[1]], X[tr[2]]], 1)
                nf = np.cross(T[:, 1] - T[:, 0], T[:, 2] - T[:, 0])
                flip = ((T.mean(1) - cin) * nf).sum(1) < 0   # normal hacia afuera
                T[flip] = T[flip][:, ::-1]
                tris.append(T)
    if not tris:
        return np.zeros((0, 3)), np.zeros((0, 3), int)
    V = np.concatenate(tris, 0).reshape(-1, 3)
    key = np.round(V * 1e9).astype(np.int64)                 # soldar vertices
    _, first, inv = np.unique(key, axis=0, return_index=True, return_inverse=True)
    return V[first], inv.reshape(-1, 3)

def vertex_normals(V, T):
    N = np.zeros_like(V)
    fn = np.cross(V[T[:, 1]] - V[T[:, 0]], V[T[:, 2]] - V[T[:, 0]])
    for k in range(3):
        np.add.at(N, T[:, k], fn)
    return N / np.maximum(np.linalg.norm(N, axis=1, keepdims=True), 1e-12)

def mesh_stats(V, T):
    """(#V, #E, #T, caracteristica de Euler, genero si es cerrada y conexa)."""
    E = np.vstack([T[:, [0, 1]], T[:, [1, 2]], T[:, [2, 0]]])
    E = len(np.unique(np.sort(E, axis=1), axis=0))
    chi = len(V) - E + len(T)
    return len(V), E, len(T), chi, (2 - chi) // 2

def make_grid(P, res=64, pad=0.12):
    lo, hi = P.min(0), P.max(0)
    c, d = (lo + hi) / 2, (hi - lo).max()
    lo, hi = c - d * (0.5 + pad), c + d * (0.5 + pad)
    axes = [np.linspace(lo[i], hi[i], res) for i in range(3)]
    Q = np.stack(np.meshgrid(*axes, indexing='ij'), -1).reshape(-1, 3)
    return axes, Q

In [ ]:
import time
# Verificacion 1: esfera de radio 1 -> area 4 pi, chi = 2
ax_s = [np.linspace(-1.5, 1.5, 48)] * 3
Gs = np.linalg.norm(np.stack(np.meshgrid(*ax_s, indexing='ij'), -1), axis=-1) - 1.0
Vs, Ts = marching_tets(Gs, ax_s)
area = 0.5 * np.linalg.norm(np.cross(Vs[Ts[:, 1]] - Vs[Ts[:, 0]],
                                     Vs[Ts[:, 2]] - Vs[Ts[:, 0]]), axis=1).sum()
nv, ne, nt, chi, gen = mesh_stats(Vs, Ts)
print('esfera: V=%d E=%d T=%d  chi=%d (esperado 2)  area=%.4f (4pi=%.4f)'
      % (nv, ne, nt, chi, area, 4 * np.pi))
assert chi == 2 and abs(area - 4 * np.pi) / (4 * np.pi) < 0.01
assert np.abs(np.linalg.norm(Vs, axis=1) - 1).max() < 0.01

# Verificacion 2: nuestra superficie, muestreando F exactamente en la grilla
axes, Qg = make_grid(P_gt, res=64)
t0 = time.time(); Vb, Tb = marching_tets(F(Qg).reshape(64, 64, 64), axes)
nv, ne, nt, chi_b, gen_b = mesh_stats(Vb, Tb)
print('blob (F exacta, 64^3): %.1f s   V=%d T=%d  chi=%d  genero=%d'
      % (time.time() - t0, nv, nt, chi_b, gen_b))
assert chi_b == 0, 'la superficie tiene genero 1, asi que chi debe ser 0'
print('error contra la superficie verdadera: mediana %.2e'
      % np.median(dist_to_surface(Vb)))

from mpl_toolkits.mplot3d.art3d import Poly3DCollection
fig = plt.figure(figsize=(9.5, 4.4), constrained_layout=True)
ax0 = fig.add_subplot(1, 2, 1)
shade(ax0, Vb, vertex_normals(Vb, Tb), s=1.2, title='malla de referencia')
ax3 = fig.add_subplot(1, 2, 2, projection='3d')
box, rad = np.array([0.95, 0.0, 0.42]), 0.22        # ventana de zoom
sel = np.all(np.abs(Vb[Tb].mean(1) - box) < rad, axis=1)
ax3.add_collection3d(Poly3DCollection(Vb[Tb[sel]], facecolor='0.85',
                                      edgecolor='0.3', linewidth=0.4))
for setlim, c in [(ax3.set_xlim, 0), (ax3.set_ylim, 1), (ax3.set_zlim, 2)]:
    setlim(box[c] - rad, box[c] + rad)
ax3.set_box_aspect((1, 1, 1)); ax3.view_init(28, 35); ax3.set_axis_off()
ax3.set_title('teselado (detalle: %d triángulos)' % sel.sum(), fontsize=10)
plt.show()

La característica de Euler es 0, o sea género 1: el algoritmo recuperó el agujero sin que
nadie se lo dijera. Esa es la ventaja de la formulación implícita.

En el detalle se ve el patrón regular de marching tetrahedra: triángulos alargados,
alineados con la descomposición del vóxel. Marching cubes produce menos triángulos y mejor
formados; el precio es la tabla de 256 casos.

---
## 2.3 Reconstrucción de Hoppe: plano tangente del punto más cercano

El primer método (Hoppe et al., SIGGRAPH 1992) construye $f$ de la forma más directa
posible. Para cada punto de consulta $\mathbf{x}$:

1. buscar la muestra más cercana $\mathbf{p}_{i(\mathbf{x})}$,
2. devolver la distancia con signo al **plano tangente** en esa muestra:

$$f(\mathbf{x}) \;=\; \mathbf{n}_{i(\mathbf{x})}^\top \left(\mathbf{x} - \mathbf{p}_{i(\mathbf{x})}\right).$$

Es una aproximación local de primer orden a la distancia con signo. Es rapidísima (una
consulta al kd-tree por punto de grilla) y muy exacta con datos limpios. Sus debilidades:

* es **discontinua**: $f$ salta al cambiar el punto más cercano, así que hereda todo el
  ruido de las normales;
* **no promedia** nada: cada valor depende de una sola muestra;
* depende críticamente de que las normales estén bien orientadas.

### Ejercicio 8 — SDF por plano tangente (Hoppe)

In [ ]:
def hoppe_sdf(Q, P, N):
    _, j = cKDTree(P).query(Q)
    return ((Q - P[j]) * N[j]).sum(1)

In [ ]:
# --- Tests del ejercicio 8 ---
res = 64
axes, Qg = make_grid(P_gt, res=res)

f_on = hoppe_sdf(P_in, P_in, N_in)
assert np.abs(f_on).max() < 1e-9, 'f debe anularse sobre las propias muestras'
assert hoppe_sdf(np.array([[0., 0., 0.]]), P_in, N_in)[0] > 0, \
    'el centro del anillo esta FUERA de la superficie'
assert hoppe_sdf(np.array([[0.9, 0., 0.]]), P_in, N_in)[0] < 0, \
    'el centro de una bola esta DENTRO'

t0 = time.time()
G_h = hoppe_sdf(Qg, P_in, N_in).reshape(res, res, res)
V_h, T_h = marching_tets(G_h, axes)
e_h = dist_to_surface(V_h)
print('Hoppe: %.1f s  V=%d T=%d  chi=%d  genero=%d' %
      ((time.time() - t0, len(V_h), len(T_h)) + mesh_stats(V_h, T_h)[3:5]))
print('error: mediana %.4f  p95 %.4f  (%.2f%% de la diagonal)'
      % (np.median(e_h), np.percentile(e_h, 95), 100 * np.median(e_h) / DIAG))
assert np.median(e_h) < 0.01 * DIAG
print('OK')

# Que pasa si las normales NO estan orientadas
rng = np.random.default_rng(2)
N_bad = N_in * rng.choice([-1.0, 1.0], len(N_in))[:, None]
V_b, T_b = marching_tets(hoppe_sdf(Qg, P_in, N_bad).reshape(res, res, res), axes)
print('con normales sin orientar: T=%d  chi=%d  <-- basura'
      % (len(T_b), mesh_stats(V_b, T_b)[3]))

fig, axs = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)
shade(axs[0], V_h, vertex_normals(V_h, T_h), s=1.2, title='Hoppe')
im = shade(axs[1], V_h, vertex_normals(V_h, T_h), s=1.2, c=e_h / DIAG,
           cmap='inferno', vmin=0, vmax=0.004, title='error / diagonal')
plt.colorbar(im, ax=axs[1], shrink=0.7)
shade(axs[2], V_b, vertex_normals(V_b, T_b), s=1.2,
      title='normales sin orientar')
plt.show()

## 2.4 Superficies implícitas por RBF

Segunda familia (Carr et al., SIGGRAPH 2001): en vez de una fórmula local, ajustar una
**función global** como combinación de funciones de base radial,

$$f(\mathbf{x}) \;=\; \sum_{j=1}^{n} w_j\, \varphi\!\left(\lVert \mathbf{x}-\mathbf{x}_j\rVert\right) \;+\; \pi(\mathbf{x}),$$

donde $\varphi(r) = r$ es el spline biarmónico en 3D (el que minimiza la energía de flexión)
y $\pi$ es un polinomio de grado 1 que absorbe la parte afín.

El detalle clave: si sólo imponemos $f(\mathbf{p}_i)=0$, la solución trivial $f\equiv 0$
gana. Hay que agregar **restricciones fuera de la superficie**, y para eso sirven las
normales: se generan puntos desplazados $\mathbf{p}_i \pm \varepsilon\,\mathbf{n}_i$ con
valores $\pm\varepsilon$. Con $3n$ restricciones el sistema queda

$$\begin{pmatrix} A & Q \\ Q^\top & 0\end{pmatrix}\begin{pmatrix}\mathbf{w}\\ \mathbf{c}\end{pmatrix} = \begin{pmatrix}\mathbf{d}\\ \mathbf{0}\end{pmatrix},
\qquad A_{ij} = \varphi(\lVert\mathbf{x}_i-\mathbf{x}_j\rVert), \qquad Q = [\,\mathbf{1}\;\; X\,] \in \mathbb{R}^{3n\times 4}.$$

Las filas $Q^\top \mathbf{w} = 0$ son las condiciones de ortogonalidad que hacen el problema
bien puesto. Es un sistema **denso**: $O(n^3)$ para resolver y $O(n)$ por evaluación. Por
eso Carr et al. dedican medio paper al *fast multipole method*; nosotros vamos a submuestrear.

### Ejercicio 9 — Sistema de RBF biarmónicas (Carr et al.)

In [ ]:
def rbf_fit(P, N, eps, reg=1e-8):
    X = np.vstack([P, P + eps * N, P - eps * N])
    d = np.concatenate([np.zeros(len(P)), np.full(len(P), eps),
                        np.full(len(P), -eps)])
    n = len(X)
    Qp = np.hstack([np.ones((n, 1)), X])
    M = np.zeros((n + 4, n + 4))
    M[:n, :n] = cdist(X, X) + reg * np.eye(n)
    M[:n, n:] = Qp
    M[n:, :n] = Qp.T
    sol = np.linalg.solve(M, np.concatenate([d, np.zeros(4)]))
    return X, sol[:n], sol[n:]

def rbf_eval(Q, X, w, c, chunk=20000):
    out = np.empty(len(Q))
    for s in range(0, len(Q), chunk):
        q = Q[s:s + chunk]
        out[s:s + chunk] = cdist(q, X) @ w + c[0] + q @ c[1:]
    return out

In [ ]:
# --- Tests del ejercicio 9 ---
sub = np.random.default_rng(0).permutation(len(P_in))[:700]
t0 = time.time(); X, w, c = rbf_fit(P_in[sub], N_in[sub], eps=0.02)
print('ajuste con %d centros (%d incognitas): %.2f s' % (len(sub), len(X) + 4,
                                                         time.time() - t0))
f0 = rbf_eval(P_in[sub], X, w, c)
fp = rbf_eval(P_in[sub] + 0.02 * N_in[sub], X, w, c)
print('|f| sobre la superficie: max %.2e   f en +eps n: media %.4f (eps=0.02)'
      % (np.abs(f0).max(), fp.mean()))
assert np.abs(f0).max() < 1e-5, 'las restricciones on-surface no se cumplen'
assert abs(fp.mean() - 0.02) < 5e-3, 'las restricciones off-surface no se cumplen'

t0 = time.time(); G_r = rbf_eval(Qg, X, w, c).reshape(res, res, res)
V_r, T_r = marching_tets(G_r, axes)
e_r = dist_to_surface(V_r)
print('RBF: evaluacion en %d^3 en %.1f s  chi=%d genero=%d' %
      ((res, time.time() - t0) + mesh_stats(V_r, T_r)[3:5]))
print('error: mediana %.4f  p95 %.4f  (%.2f%% de la diagonal)'
      % (np.median(e_r), np.percentile(e_r, 95), 100 * np.median(e_r) / DIAG))
assert np.median(e_r) < 0.01 * DIAG
print('OK')

# Costo cubico: medir el tiempo de ajuste al crecer n
print('\ncosto del ajuste:')
for n in [200, 400, 800]:
    s = np.random.default_rng(0).permutation(len(P_in))[:n]
    t0 = time.time(); rbf_fit(P_in[s], N_in[s], 0.02); dt = time.time() - t0
    print('  n=%4d  (%5d incognitas)  %.2f s' % (n, 3 * n + 4, dt))

fig, axs = plt.subplots(1, 2, figsize=(9.5, 4.4), constrained_layout=True)
shade(axs[0], V_r, vertex_normals(V_r, T_r), s=1.2,
      title='RBF, 700 centros')
im = shade(axs[1], V_r, vertex_normals(V_r, T_r), s=1.2, c=e_r / DIAG,
           cmap='inferno', vmin=0, vmax=0.004, title='error / diagonal')
plt.colorbar(im, ax=axs[1], shrink=0.7); plt.show()

El error de RBF con 700 centros no está donde está el ruido, sino donde falta resolución:
la solución es global y suave, y con pocos centros no puede seguir los detalles finos. Subir
el número de centros mejora el ajuste pero cuesta $O(n^3)$.

Sobre $\varepsilon$: si es muy grande, los puntos desplazados se meten al otro lado de la
superficie en zonas de alta curvatura y el ajuste se rompe; si es muy chico, las tres
restricciones por punto quedan casi colineales y el sistema se mal condiciona. La regla
práctica de Carr et al. es $\varepsilon \approx$ 1% del diámetro del modelo, verificando que
el punto desplazado siga teniendo a $\mathbf{p}_i$ como vecino más cercano.

---
## 2.5 Reconstrucción de Poisson

Tercera familia (Kazhdan, Bolitho & Hoppe, SGP 2006), y la que se usa hoy en día. La idea
es más elegante que las dos anteriores.

**Observación de partida.** Sea $\chi$ la *función indicadora* del sólido (1 adentro, 0
afuera). Su gradiente es cero en todas partes salvo en la superficie, donde vale una delta
apuntando **hacia adentro**. Es decir: si tomamos las normales de la nube y las "untamos"
sobre una grilla, obtenemos un campo vectorial $\vec{V}$ que **debería coincidir con el
gradiente de la indicadora** (salvo signo y suavizado):

$$\nabla \chi \;\approx\; \vec{V}.$$

Ese sistema está sobredeterminado (3 ecuaciones por incógnita), así que no se resuelve
directamente: se toma divergencia a ambos lados y queda una **ecuación de Poisson**,

$$\Delta \chi \;=\; \nabla\cdot\vec{V},$$

que sí tiene solución única (salvo constante) y es un problema estándar. Después se extrae
la isosuperficie en un valor $\gamma$ estimado como el promedio de $\chi$ evaluada en las
propias muestras:

$$\gamma = \frac{1}{n}\sum_i \chi(\mathbf{p}_i).$$

**Por qué funciona tan bien.** El operador $\Delta^{-1}$ es un suavizador global: promedia
la información de *todos* los puntos, así que el ruido se cancela y los huecos se rellenan
por continuidad. A diferencia de Hoppe, ningún valor depende de una sola muestra; a
diferencia de RBF, el sistema es **disperso** (o resoluble por FFT), no denso.

**Nuestra versión.** Kazhdan usa un octree y una base de B-splines. Nosotros usamos una
grilla regular, splatting trilineal de las normales, diferencias finitas para la divergencia
y **FFT** para invertir el laplaciano. Con normales hacia afuera, $\vec{V} = \nabla(-\chi)$,
así que la $\chi$ que obtenemos es negativa adentro y positiva afuera — la misma convención
de signo que veníamos usando.

El laplaciano discreto de 7 puntos tiene símbolo de Fourier
$\hat{L}(\mathbf{k}) = \frac{2}{h^2}\sum_{d=1}^{3}\left(\cos k_d - 1\right)$, así que en el
dominio de Fourier la ecuación se resuelve dividiendo, con el modo constante puesto a cero.

In [ ]:
def splat_normals(P, N, axes):
    """Reparte cada normal en los 8 nodos vecinos con pesos trilineales.
    Devuelve V de forma (3, res, res, res).  (Se entrega hecha.)"""
    res = len(axes[0])
    h = np.array([a[1] - a[0] for a in axes]); lo = np.array([a[0] for a in axes])
    g = (P - lo) / h
    i0 = np.floor(g).astype(int); t = g - i0
    V = np.zeros((3, res, res, res))
    for dx in (0, 1):
        for dy in (0, 1):
            for dz in (0, 1):
                wgt = ((t[:, 0] if dx else 1 - t[:, 0]) *
                       (t[:, 1] if dy else 1 - t[:, 1]) *
                       (t[:, 2] if dz else 1 - t[:, 2]))
                idx = i0 + np.array([dx, dy, dz])
                ok = np.all((idx >= 0) & (idx < res), 1)
                ii, jj, kk = idx[ok].T
                for c in range(3):
                    np.add.at(V[c], (ii, jj, kk), wgt[ok] * N[ok, c])
    return V

def smooth_field(V, h, sigma_vox=1.0):
    """Suavizado gaussiano del campo vectorial, por FFT.  (Se entrega hecha.)"""
    res = V.shape[1]
    k = np.fft.fftfreq(res) * 2 * np.pi / h
    K2 = k[:, None, None] ** 2 + k[None, :, None] ** 2 + k[None, None, :] ** 2
    G = np.exp(-0.5 * (sigma_vox * h) ** 2 * K2)
    return np.stack([np.real(np.fft.ifftn(np.fft.fftn(V[c]) * G)) for c in range(3)])

def trilinear(G, axes, Q):
    """Interpola el campo escalar G en los puntos Q.  (Se entrega hecha.)"""
    res = len(axes[0])
    h = np.array([a[1] - a[0] for a in axes]); lo = np.array([a[0] for a in axes])
    g = (Q - lo) / h
    i0 = np.clip(np.floor(g).astype(int), 0, res - 2); t = g - i0
    out = np.zeros(len(Q))
    for dx in (0, 1):
        for dy in (0, 1):
            for dz in (0, 1):
                wgt = ((t[:, 0] if dx else 1 - t[:, 0]) *
                       (t[:, 1] if dy else 1 - t[:, 1]) *
                       (t[:, 2] if dz else 1 - t[:, 2]))
                out += wgt * G[i0[:, 0] + dx, i0[:, 1] + dy, i0[:, 2] + dz]
    return out

### Ejercicio 10 — Divergencia, solver de Poisson por FFT y el iso-valor

In [ ]:
def divergence(V, h):
    return sum(np.gradient(V[c], h, axis=c, edge_order=2) for c in range(3))

def solve_poisson_fft(rhs, h):
    res = rhs.shape[0]
    k = 2 * np.pi * np.fft.fftfreq(res)
    L = (2 * np.cos(k) - 2) / h ** 2
    D = L[:, None, None] + L[None, :, None] + L[None, None, :]
    D[0, 0, 0] = 1.0
    f = np.fft.fftn(rhs) / D
    f[0, 0, 0] = 0.0
    return np.real(np.fft.ifftn(f))

def poisson_reconstruct(P, N, res=64, pad=0.12, sigma_vox=1.0):
    axes, _ = make_grid(P, res, pad)
    h = axes[0][1] - axes[0][0]
    V = smooth_field(splat_normals(P, N, axes), h, sigma_vox)
    chi = solve_poisson_fft(divergence(V, h), h)
    iso = trilinear(chi, axes, P).mean()
    return axes, chi, iso

In [ ]:
# --- Tests del ejercicio 10 ---
# --- Test 1: el solver, contra una solucion analitica periodica
n = 32; L = 1.0; h = L / n
x = np.arange(n) * h
XX, YY, ZZ = np.meshgrid(x, x, x, indexing='ij')
chi_true = np.sin(2 * np.pi * XX) * np.sin(2 * np.pi * YY) * np.sin(2 * np.pi * ZZ)
lap = np.zeros_like(chi_true)                       # laplaciano discreto exacto
for c in range(3):
    lap += (np.roll(chi_true, 1, c) - 2 * chi_true + np.roll(chi_true, -1, c)) / h ** 2
chi_num = solve_poisson_fft(lap, h)
err = np.abs(chi_num - chi_true).max()
print('solver: error maximo contra la solucion analitica = %.2e' % err)
assert err < 1e-10, 'revisa el simbolo del laplaciano o el modo DC'

# --- Test 2: la divergencia
Vt = np.stack([XX, np.zeros_like(XX), 2 * ZZ])      # div = 1 + 2 = 3
d = divergence(Vt, h)
assert abs(np.median(d) - 3.0) < 1e-6, 'div([x,0,2z]) = 3'

# --- Test 3: reconstruccion completa
t0 = time.time()
ax_p, chi, iso = poisson_reconstruct(P_in, N_in, res=res)
V_p, T_p = marching_tets(chi, ax_p, iso=iso)
e_p = dist_to_surface(V_p)
print('\nPoisson: %.1f s  iso=%.4f  V=%d T=%d  chi=%d  genero=%d'
      % ((time.time() - t0, iso, len(V_p), len(T_p)) + mesh_stats(V_p, T_p)[3:5]))
print('error: mediana %.4f  p95 %.4f  (%.2f%% de la diagonal)'
      % (np.median(e_p), np.percentile(e_p, 95), 100 * np.median(e_p) / DIAG))
assert mesh_stats(V_p, T_p)[3] == 0, 'la topologia deberia ser de genero 1'
assert np.median(e_p) < 0.005 * DIAG
print('OK')

fig, axs = plt.subplots(1, 3, figsize=(13.5, 4.2), constrained_layout=True)
sl = chi[:, :, res // 2]
im = axs[0].imshow(sl.T, origin='lower', cmap='RdBu_r',
                   vmin=-np.abs(sl).max(), vmax=np.abs(sl).max())
axs[0].contour(sl.T, levels=[iso], colors='k', linewidths=1.5)
axs[0].set_title(r'$\chi$ en un corte, y el nivel $\gamma$', fontsize=10)
axs[0].axis('off'); plt.colorbar(im, ax=axs[0], shrink=0.75)
shade(axs[1], V_p, vertex_normals(V_p, T_p), s=1.2, title='Poisson')
im = shade(axs[2], V_p, vertex_normals(V_p, T_p), s=1.2, c=e_p / DIAG,
           cmap='inferno', vmin=0, vmax=0.004, title='error / diagonal')
plt.colorbar(im, ax=axs[2], shrink=0.7); plt.show()

En el corte se ve por qué el método es robusto: $\chi$ es una meseta suave que cambia de
signo en la superficie, no un campo que dependa punto a punto de la muestra más cercana.

Un detalle de nuestra implementación: las condiciones de borde periódicas de la FFT son
legítimas aquí porque $\vec{V}$ tiene soporte compacto dentro de la grilla (por eso el
*padding* del 12%). La implementación de Kazhdan usa Dirichlet sobre un octree adaptativo,
que además concentra la resolución cerca de la superficie.

## 2.6 Comparación en los tres regímenes difíciles

Los tres métodos dan prácticamente lo mismo con datos limpios y densos. Las diferencias
aparecen con ruido, con huecos y con baja densidad.

In [ ]:
def reconstruct_all(P, N, res=64):
    out = {}
    axes, Qg = make_grid(P_gt, res=res)          # misma grilla para comparar
    V, T = marching_tets(hoppe_sdf(Qg, P, N).reshape(res, res, res), axes)
    out['Hoppe'] = (V, T)
    s = np.random.default_rng(0).permutation(len(P))[:700]
    X, w, c = rbf_fit(P[s], N[s], eps=0.02)
    V, T = marching_tets(rbf_eval(Qg, X, w, c).reshape(res, res, res), axes)
    out['RBF'] = (V, T)
    ax_p, chi, iso = poisson_reconstruct(P, N, res=res)
    out['Poisson'] = marching_tets(chi, ax_p, iso=iso)
    return out

def report(tag, out):
    for k, (V, T) in out.items():
        e = dist_to_surface(V)
        print('  %-8s  err mediana %6.4f  p95 %6.4f  |  T=%6d  chi=%3d  genero=%d'
              % ((k, np.median(e), np.percentile(e, 95), len(T))
                 + mesh_stats(V, T)[3:5]))

casos = {}
# (a) ruido fuerte
Pa = add_noise(P_gt, N_gt, 0.02, seed=9)
Na = orient_normals(Pa, normals_pca(Pa, k=60)[0], k=12)
casos['ruido sigma=0.02'] = (Pa, Na)
# (b) hueco: se elimina un casquete (escaneo parcial)
m = np.linalg.norm(P_gt - np.array([0.9, 0.0, 0.35]), axis=1) > 0.8
casos['hueco (%d pts menos)' % (~m).sum()] = (P_gt[m], N_gt[m])
# (c) baja densidad
idx = np.random.default_rng(0).permutation(len(P_gt))[:1200]
casos['densidad baja (1200 pts)'] = (P_gt[idx], N_gt[idx])

resultados = {}
for tag, (Pc, Nc) in casos.items():
    print(tag)
    resultados[tag] = reconstruct_all(Pc, Nc)
    report(tag, resultados[tag])

In [ ]:
fig, axs = plt.subplots(3, 4, figsize=(14, 10.5), constrained_layout=True)
for i, (tag, (Pc, Nc)) in enumerate(casos.items()):
    shade(axs[i, 0], Pc, Nc, s=4, title='entrada: ' + tag)
    for j, k in enumerate(['Hoppe', 'RBF', 'Poisson']):
        V, T = resultados[tag][k]
        shade(axs[i, j + 1], V, vertex_normals(V, T), s=1.2,
              title='%s  (genus %d)' % (k, mesh_stats(V, T)[4]))
plt.show()

Lo que hay que leer de la tabla y las figuras:

* **Con ruido**, Hoppe se degrada mucho más que los otros dos y, sobre todo, **rompe la
  topología**: aparecen asas y componentes espurias (característica de Euler muy negativa).
  Es la consecuencia directa de que $f$ dependa de una sola muestra. Poisson mantiene género
  1 y el error casi no sube.
* **Con un hueco**, Hoppe es exactísimo donde sí hay datos (mediana ~0.0003) pero rompe la
  topología en el borde: el plano tangente de las muestras del contorno se extiende hacia
  adentro y genera superficie espuria (género 4 en vez de 1). RBF y Poisson cierran el hueco
  con un parche suave y mantienen la topología; el p95 mide cuánto se aparta ese parche de
  la superficie verdadera, y ahí RBF (~0.04) le gana holgadamente a nuestro Poisson (~0.22):
  el spline biarmónico extrapola con continuidad de segundo orden, mientras que dentro del
  hueco el campo $\vec V$ es cero y la solución de Poisson sólo puede ser armónica, es decir
  lo más plana posible. En cualquier caso ninguno de los dos avisa: rellenan sin distinguir
  dato de interpolación, que es justo lo que no se puede hacer en patrimonio (ver el recorte
  por densidad del Bloque 3).
* **Con baja densidad**, Hoppe y RBF andan mejor que Poisson, porque Poisson está limitado
  por la resolución de la grilla y por el suavizado, no por los datos.

## Point2Mesh y los métodos neuronales

Las slides cierran con Point2Mesh (Hanocka et al., SIGGRAPH 2020), que cambia el
planteamiento: en vez de ajustar un campo escalar, **deforma una malla inicial** (la
envolvente convexa) hasta que se ajusta a la nube, usando una red convolucional sobre
mallas (MeshCNN) cuyos pesos se optimizan **para esa forma en particular** — no hay
entrenamiento previo ni dataset.

El argumento es el *self-prior*: la propia arquitectura convolucional, al compartir pesos
entre regiones de la malla, favorece explicaciones donde las partes se repiten. Como las
formas reales tienen mucha auto-similitud y el ruido no, la red ajusta primero la señal y
después el ruido — si se detiene a tiempo, se obtiene reconstrucción y denoising a la vez.
Además la malla resultante es cerrada y sin auto-intersecciones **por construcción**, porque
se parte de una malla válida y sólo se deforma.

En la misma línea están los métodos de campo neuronal (DeepSDF, occupancy networks, NKSR):
reemplazan la grilla o las RBF por una red que representa $f$, con el mismo esquema de
"campo implícito + extracción de isosuperficie" que vimos aquí. La segunda flecha del
pipeline —marching cubes— sigue siendo la misma de 1987.

---
# Cierre

Lo que quedó montado, de principio a fin:

| Paso | Lo que implementamos | Referencia |
|---|---|---|
| Ajuste de plano local | TLS = autovector menor de la covarianza (Lagrange) | Hoppe et al. 1992 |
| Normales | PCA local vectorizado, barrido de $k$, $k^*\propto\sigma^{2/3}$ | Mitra & Nguyen 2003 |
| Orientación | MST con peso $1-\lvert n_i^\top n_j\rvert$ + recorrido BFS | Hoppe et al. 1992 |
| Outliers | filtro estadístico + variación de superficie $\lambda_0/\sum\lambda$ | Pauly et al. 2002 |
| Isosuperficie | marching squares y marching tetrahedra | Lorensen & Cline 1987; Doi & Koide 1991 |
| Campo escalar (a) | plano tangente del vecino más cercano | Hoppe et al. 1992 |
| Campo escalar (b) | RBF biarmónicas con restricciones off-surface | Carr et al. 2001 |
| Campo escalar (c) | $\Delta\chi = \nabla\cdot\vec V$ resuelto por FFT | Kazhdan et al. 2006 |

## Preguntas de discusión

1. En el barrido del ejercicio 3, la rama de ruido cae como $k^{-1}$ y la de curvatura sube
   como $k^{1/2}$. ¿Qué cambia si el ruido no es a lo largo de la normal sino isotrópico?
   ¿Y si la densidad de muestreo no es uniforme?
2. El MST del ejercicio 4 orienta bien esta superficie. Construya un caso donde falle
   (pista: dos hojas de superficie paralelas y cercanas, con muestreo escaso). ¿Qué
   información adicional habría que usar?
3. La variación de superficie $\sigma_\lambda$ detecta outliers *y* mide curvatura. ¿Cómo se
   distinguen ambos efectos con una sola vecindad? ¿Y con dos escalas?
4. Marching tetrahedra no tiene casos ambiguos, pero su malla es peor. ¿Vale la pena en una
   aplicación real, o conviene invertir en la tabla de 256 casos de marching cubes?
5. En marching cubes, ¿por qué elegir mal la resolución de la grilla puede cambiar el
   **género** de la malla reconstruida? Relacione esto con el `alcance` (*reach*) de la
   superficie.
6. Poisson rellena los huecos sin avisar. En un contexto de patrimonio cultural, donde hay
   que distinguir lo medido de lo interpolado, ¿qué agregaría al pipeline?
7. RBF es global y $O(n^3)$; Poisson es local y casi lineal. ¿En qué régimen de datos
   (cantidad, ruido, huecos) preferiría igualmente RBF?
8. El iso-valor $\gamma$ se estimó como el promedio de $\chi$ en las muestras. ¿Qué pasa si
   la nube tiene densidad muy desigual? ¿Cómo lo corregiría?
9. Point2Mesh no usa datos de entrenamiento: el *prior* está en la arquitectura. ¿Qué
   evidencia pediría para creer que ese prior es realmente el que explica el resultado, y no
   simplemente el suavizado de la malla inicial?

## Bibliografía

* H. Hoppe, T. DeRose, T. Duchamp, J. McDonald, W. Stuetzle. *Surface Reconstruction from
  Unorganized Points.* SIGGRAPH 1992.
* W. Lorensen, H. Cline. *Marching Cubes: A High Resolution 3D Surface Construction
  Algorithm.* SIGGRAPH 1987.
* G. Nielson, B. Hamann. *The Asymptotic Decider: Resolving the Ambiguity in Marching
  Cubes.* IEEE Visualization 1991.
* A. Doi, A. Koide. *An Efficient Method of Triangulating Equi-valued Surfaces by Using
  Tetrahedral Cells.* IEICE Trans. 1991. (marching tetrahedra)
* J. C. Carr et al. *Reconstruction and Representation of 3D Objects with Radial Basis
  Functions.* SIGGRAPH 2001.
* M. Kazhdan, M. Bolitho, H. Hoppe. *Poisson Surface Reconstruction.* SGP 2006.
* M. Kazhdan, H. Hoppe. *Screened Poisson Surface Reconstruction.* ACM TOG 2013.
* N. J. Mitra, A. Nguyen. *Estimating Surface Normals in Noisy Point Cloud Data.* SoCG 2003.
* M. Pauly, M. Gross, L. Kobbelt. *Efficient Simplification of Point-Sampled Surfaces.*
  IEEE Visualization 2002.
* R. Hanocka, G. Metzer, R. Giryes, D. Cohen-Or. *Point2Mesh: A Self-Prior for Deformable
  Meshes.* SIGGRAPH 2020.
* M. Berger et al. *A Survey of Surface Reconstruction from Point Clouds.* Computer Graphics
  Forum 2017.

---
# Soluciones

Cada celda de abajo está plegada: ejecútala para redefinir la función correspondiente. Haz
doble clic sobre el título para ver el código.